# **TP2 | Premiers pas avec les entiers**

*Compétences mathématiques :*
- Manipulation de (grands) entiers
- Sensibilisation aux risques des flottants

*Compétences cryptographiques :*
- Révision de l'algorithme Diffie-Hellman
- Ordres de grandeur des tailles de clés

*Compétences en programmation :*
- Génération de paramètres Diffie-Hellman avec OpenSSL
- Conversions entiers - `bytes`
- Lecture de fichiers
- Génération de clés privées et publiques

## 2.1. Conjecture de Syracuse

Un problème bien connu en arithmétique concerne la suite de Syracuse, qui prend la forme 
$$ 
u_{n+1} = \begin{cases}
    u_n / 2 & \text{~si~} u_n \text{~est pair}, \\
    3 u_n + 1 & \text{~si~} u_n \text{~est impair}.
\end{cases}
$$
avec un certain point de départ $u_0 = a \in \mathbb{N}^*$. Par exemple avec un nombre de départ de $5$ ou $6$, on a 

|          n         | 0 |  1 |  2 |  3 |  4 |  5     |  6 |  7 |  8     |  9 |
|--------------------|---|----|----|----|----|--------|----|----|--------|----|
| $u_n$ avec $a = 5$ | 5 | 16 |  8 |  4 |  2 |  **1** |  4 |  2 |  1     |  4 |
| $u_n$ avec $a = 6$ | 6 |  3 | 10 |  5 | 16 |  8     |  4 |  2 |  **1** |  4 |

On observe que la suite finit toujours par tomber sur un cycle $4, 2, 1$. Pour identifier ce cycle, on dénote $N(a)$ le plus petit indice tel que $u_{N(a)} = 1$ avec $u_0 = a$, i.e. 
$$ N(a) = \min \{n \in \mathbb{N} \mid u_n = 1, u_0 = a\} . $$
Ce nombre s'appelle la *durée de vol* associée à $a$.

En vérité, on n'a pas de garantie que $N(a)$ existe, il existe peut-être des nombres initiaux pour lesquels la suite ne retombe jamais sur $1$. Mais tous ont été testé jusqu'à $2^{68} \approx 3 \times 10^{20}$ ([source](https://doi.org/10.1007%2Fs11227-020-03368-x), merci Wikipédia), donc on n'aura pas de souci pour ce TP. En apparence pourtant simple, ce problème reste ouvert depuis au moins deux siècles. Une [vidéo de El Jj](https://www.youtube.com/watch?v=BP2G28694z8) présente brièvement le problème, son histoire et des résultats associés.

### 2.1.a. La fonction de Syracuse

Coder une fonction `syracuse_fun` qui permet d'obtenir $u_{n+1}$ à partir de $u_n$. Tester cette fonction avec des valeurs paires et impaires choisies.

In [ ]:
def syracuse_fun(un):
    """
    Cette fonction prend un entier `un` en entrée et renvoie le terme suivant dans la séquence de Syracuse.

    Args:
        un (int): Le terme actuel dans la séquence de Syracuse.

    Returns:
        int: Le terme suivant dans la séquence de Syracuse.
    """
    if un % 2 == 0:
        return un // 2  # <- quotient de division euclidienne
    else:
        return 3 * un + 1

### 2.1.b. Calcul d'une durée de vol

**TODO**
- Coder une fonction `duree_syracuse` qui calcule la durée de vol associée à une donnée initiale $a$. 
- Vérifier qu'avec $a = 5$ et $a = 6$, on obtient bien $N(5) = 5$ et $N(6) = 8$.

In [ ]:
def duree_syracuse(a: int):
    """
    Calcule le nombre d'itérations nécessaires pour que la suite de Syracuse à partir de l'entier `a` atteigne la valeur 1.
    
    Args:
        a (int): L'entier à partir duquel la suite de Syracuse est calculée.

    Returns:
        int: Le nombre d'itérations nécessaires pour atteindre la valeur 1.
    """
    if a < 1:
        raise ValueError("La suite ne «vole» qu'à partir d'un entier strictement positif.")

    un = a
    n = 0

    while un != 1:
        un = syracuse_fun(un)
        n += 1
    return n

## 2.2. L'importance d'utiliser des entiers

Dans cette partie, on va se rendre compte de l'importance de n'utiliser **que** des entiers pour manipuler de grands nombres. 

Pour cela, on va manipuler des nombres de plusieurs centaines de bits et comparer les résultats des méthodes spécialisées avec celles qui sembleraient plus naturelles.

Dénotons $s = \lfloor \sqrt{n} \rfloor$ la partie entière de la racine carrée d'un nombre $n$. Par définition, on a 
$$ s \leq \sqrt{n} <  s + 1. $$

In [ ]:
from math import isqrt, sqrt, floor
import os

### 2.2.a. Génération de nombres

**TODO**
- Avec la fonction `os.urandom`, générer 1024 bits (128 octets) de données aléatoires.
- Convertir ces données en un entier positif `n` avec la méthode `int.from_bytes` (*big-endian*).
- Vérifier que sa taille en bits est bien proche de 1024.

In [ ]:
n_bytes = os.urandom(128)
n = int.from_bytes(n_bytes, "big")
print("Taille de `n` en bits:", n.bit_length())

### 2.2.b. Calcul de la racine carrée entière

**TODO**\
Calculer la racine carrée entière de `n` de deux manières :
- avec la fonction `isqrt`
- avec la fonction `sqrt` et la partie entière `floor`
Est-ce que les deux résultats sont identiques ?

In [ ]:
isqrt_n = isqrt(n)
sqrt_n = floor(sqrt(n))

print("isqrt(n):       ", isqrt_n)
print("floor(sqrt(n)): ", sqrt_n)
# On observe qu'après ~15 chiffres significatifs, les valeurs deviennent différentes
# c'est à cause des erreurs d'arrondi lors de la conversion `int -> float`
# implicite dans le calcul de `sqrt(n)`.

print("Les résultats sont différents:", isqrt_n != sqrt_n)

### 2.2.c. Validation de la racine carrée entière

En passant au carré, on obtient
$$ s^2 \leq n < (s + 1)^2. $$

**TODO**\
Vérifier que cette inégalité est bien respectée pour `isqrt` mais pas pour `sqrt` et `floor`.

In [ ]:
is_valid = lambda s: s**2 <= n < (s + 1) ** 2
print("isqrt(n) vérifie l'inégalité:", is_valid(isqrt_n))
print("floor(sqrt(n)) vérifie l'inégalité:", is_valid(sqrt_n))

### 2.2.d. Comparaison graphique des résultats

**TODO**
Sur 500 nombres aléatoires de 128 bits, calculer 
- la racine carrée entière de chacun de ces entiers avec les deux méthodes.
- le nombre de bits de l'erreur de la méthode `sqrt` et `floor` par rapport à la méthode `isqrt`.

Afficher cette statistique sous forme de [boîte à moustache](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.boxplot.html). De combien de bits est l'erreur moyenne ?

In [ ]:
import matplotlib.pyplot as plt

sample_size = 500
errbits = [0 for _ in range(sample_size)]
for k in range(sample_size):
    nk = int.from_bytes(os.urandom(128), "big")
    errbits[k] = (floor(sqrt(nk)) - isqrt(nk)).bit_length()

plt.boxplot(errbits);

### 2.2.e. Avec un nombre encore plus grand

**TODO**\
Générer un entier aléatoire de 2048 bits et calculer sa racine entière avec `sqrt` et `floor`. Que se passe-t-il ?

In [ ]:
n = int.from_bytes(os.urandom(256), "big")

try:
    sqrt_n = floor(sqrt(n))
except OverflowError as e:
    print("Erreur:", e)

## 2.3. Paramètres Diffie-Hellman

Cette partie et la suivante suivent essentiellement le tutoriel officiel de la bibliothèque `cryptography` : [Diffie-Hellman key exchange](https://cryptography.io/en/latest/hazmat/primitives/asymmetric/dh/).

In [ ]:
# On évite les alertes d'obsolescence :
# Diffie-Hellman (DH) est déprécié, mais on en a besoin pour la pédagogie !
import warnings
from cryptography.utils import CryptographyDeprecationWarning
warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

### 2.3.a. Générer des paramètres avec OpenSSL

**TODO**\
En utilisant la commande [`openssl genpkey`](https://docs.openssl.org/3.0/man1/openssl-genpkey/#examples), générer un fichier `dhparams.pem` de paramètres Diffie-Hellman de 2048 bits.

In [ ]:
# issu directement d'un exemple de la documentation,
# en modifiant le fichier de sortie et en ajoutant l'option `-quiet`
!openssl genpkey -genparam -algorithm DH -out dhparams.pem \
    -pkeyopt dh_paramgen_prime_len:2048 -quiet

### 2.3.b. Lecture des paramètres avec `cryptography`

**TODO**
- Extraire le modulus `p` et le générateur `g` de `params_nums`
- À l'aide de la méthode `int.bit_length`, vérifier que le modulus est bien de 2048 bits

In [ ]:
from cryptography.hazmat.primitives.serialization import load_pem_parameters

with open("dhparams.pem", "rb") as f:
    dh_data = f.read()

dh_params = load_pem_parameters(dh_data, None)
dh_params_nums = dh_params.parameter_numbers()

# extraction des paramètres
g = dh_params_nums.g
p = dh_params_nums.p

print("Générateur `g`:", g)
print("Taille du modulus `p` en bits:", p.bit_length())

## 2.4. Échange de clés avec `cryptography`



### 2.4.a. Génération de clés privées

**TODO**\
Vérifier que `xa` et `xb` sont bien différents l'un de l'autre. Quelle est leur taille en bits ? Quelle est la taille de `xa * xb` ?

In [ ]:
pkey_a = dh_params.generate_private_key()
pkey_b = dh_params.generate_private_key()

xa = pkey_a.private_numbers().x
xb = pkey_b.private_numbers().x

xx = xa * xb

print("Les nombres secrets sont différents:", xa != xb)
print("Taille des secrets:", xa.bit_length(), "(xa)   ", xb.bit_length(), "(xb)")
print("Taille de xa * xb:", xx.bit_length())

### 2.4.b. Calcul des nombres publics

**TODO**
- En utilisant la fonction `pow`, calculer les nombres publics `ya` et `yb`.
- Est-ce qu'on peut le faire avec `**` et `%` ?
- Comparer leur valeurs avec celles obtenues avec `cryptography`. Sont-elles identiques ?

In [ ]:
pubkey_a = pkey_a.public_key()
pubkey_b = pkey_b.public_key()

# TODO
# Calculer `ya`, `yb`, `za`, `zb`, `zz` avec la fonction `pow`.
# Est-ce qu'on peut le faire avec `**` et `%` ? 

# Calcul "à la main"
ya = pow(g, xa, p)
yb = pow(g, xb, p)
# Calcul avec `cryptography`
ya_crypto = pubkey_a.public_numbers().y
yb_crypto = pubkey_b.public_numbers().y
# Vérification que les deux approches coïncident
print("Nombre public cohérent pour Alice:", ya == ya_crypto)
print("Nombre public cohérent pour Bob:", yb == yb_crypto)

### 2.4.c. Échange de clés à la main

**TODO**
- Calculer les clés secrètes `za`, `zb` et `zz` avec la fonction `pow`.
- Vérifier qu'elles sont toutes égales. Quelle est leur taille en bits ?

In [ ]:
# Vérifier que `za`, `zb` et `zz` sont bien égaux. Quelle est leur taille en bits ?

za = pow(yb, xa, p)
zb = pow(ya, xb, p)
zz = pow(g, xx, p)  # xx = xa * xb

print("Tous les secrets sont identiques:", za == zb == zz)
print("Taille du secret en bits:", zz.bit_length())

### 2.4.d. Échange de clés avec `cryptography`

**TODO**
- Grâce à la méthode `exchange` de sa clé privée, calculer la clé secrète obtenue par Alice.
- Convertir cette clé en `int` (*big-endian*) et vérifier qu'elle est bien égale à `za`, `zb` et `zz`.

In [ ]:
zz_a_bytes = pkey_a.exchange(pubkey_b)
zz_a = int.from_bytes(zz_a_bytes, "big")

# pas besoin de vérifier l'égalité avec tous les nombres au-dessus vu qu'ils sont tous égaux
print("On obtient le même résultat avec la méthode `exchange`:", zz_a == zz)

## 2.5. Bonus : Conjecture de Syracuse

1. Trouver le nombre le plus haut atteint par une suite avec $u_0 \in \{1, ..., 10^4\}$ ;
2. Quitte à allouer plus de mémoire, calculer le vol le plus long pour $a \in \{1, ..., 10^6\}$ en moins de 3 secondes (il s'agit du [problème 14 du Projet Euler](https://projecteuler.net/problem=14)).